# TP53 Mutation Status Baseline

This notebook recreates the original Colab exploration in a cleaner local workflow. The goal for this first milestone is binary classification: each CCLE cell line is labeled as TP53-mutant or TP53-wild-type, then gene expression is used to rank features and train a first simple model.

## 1. Load Shared Project Code

The notebook imports the same modules used by the command-line scripts. That way, the exploratory notebook and the reproducible pipeline do not drift apart as the project grows.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tp53_baseline.config import BaselineConfig
from tp53_baseline.data import (
    TARGET_COLUMN,
    build_labeled_dataset,
    get_numeric_feature_matrix,
    get_tp53_related_columns,
    load_expression_data,
    load_mutation_data,
    summarize_dataset,
)
from tp53_baseline.feature_selection import compute_feature_ranking, exclude_columns_by_name, select_top_features
from tp53_baseline.model_training import train_top_features_model

## 2. Configure Local Paths

The raw CCLE CSVs are intentionally kept outside git because they are large. The defaults below match the local files used for the first baseline run, and each path can be overridden with environment variables.

In [ ]:
expression_path = Path(os.environ.get("TP53_EXPRESSION_CSV", "/Users/v_angelov/Downloads/CCLE_expression_full.csv"))
mutation_path = Path(os.environ.get("TP53_MUTATION_CSV", "/Users/v_angelov/Downloads/CCLE_mutations.csv"))
output_dir = Path(os.environ.get("TP53_OUTPUT_DIR", REPO_ROOT / "outputs" / "baseline"))
model_output_dir = Path(os.environ.get("TP53_MODEL_OUTPUT_DIR", REPO_ROOT / "outputs" / "models" / "top500_logistic"))
top_n = int(os.environ.get("TP53_TOP_N", "500"))
exclude_prefix = os.environ.get("TP53_EXCLUDE_PREFIX", "TP53")

expression_path, mutation_path, output_dir, model_output_dir

## 3. Inspect The Raw Tables

The expression matrix has one row per cell line and many gene-expression columns. The mutation table has one row per mutation event, so we reduce it to a per-cell-line TP53 label in the next step.

In [ ]:
expression_df = load_expression_data(expression_path)
mutation_df = load_mutation_data(mutation_path)

expression_df.shape, mutation_df.shape

In [ ]:
expression_df.head()

In [ ]:
mutation_df.head()

In [ ]:
{
    "expression_columns_preview": expression_df.columns[:10].tolist(),
    "mutation_columns": mutation_df.columns.tolist(),
    "overlapping_depmap_ids": len(set(expression_df["DepMap_ID"]).intersection(set(mutation_df["DepMap_ID"]))),
}

## 4. Build Binary TP53 Labels

A cell line is labeled `1` if it has at least one mutation row where `Hugo_Symbol == "TP53"`. All other expression samples are labeled `0` for this first binary task.

In [ ]:
tp53_labels = mutation_df[mutation_df["Hugo_Symbol"] == "TP53"][["DepMap_ID"]].drop_duplicates()
tp53_labels["TP53_status"] = 1

tp53_labels.head()

In [ ]:
dataset = build_labeled_dataset(expression_df, mutation_df)

dataset[["DepMap_ID", TARGET_COLUMN]].head()

In [ ]:
dataset[TARGET_COLUMN].value_counts().rename(index={0: "WT", 1: "TP53-mutant"})

## 5. Prepare Numeric Expression Features

We keep numeric expression columns only. Before ranking, we remove columns whose names contain `TP53`, including the direct `TP53` expression column and related TP53-family columns. This avoids giving the model an overly direct signal from the gene whose mutation status we are trying to predict.

In [ ]:
features = get_numeric_feature_matrix(dataset)
target = dataset[TARGET_COLUMN]
tp53_related_columns = get_tp53_related_columns(list(features.columns), exclude_prefix)
filtered_features = exclude_columns_by_name(features, tp53_related_columns)

summary = summarize_dataset(
    expression_df=expression_df,
    mutation_df=mutation_df,
    dataset=dataset,
    numeric_features=features,
    filtered_features=filtered_features,
    excluded_columns=tp53_related_columns,
)
summary

In [ ]:
tp53_related_columns[:20]

## 6. Rank Genes And Save The Top 500

The feature ranking follows the original Colab idea: compare mutant versus wild-type expression using absolute mean difference, absolute Cohen's d effect size, and Welch t-test p-value. For this first artifact, features are sorted by absolute effect size.

In [ ]:
ranking_df = compute_feature_ranking(filtered_features, target)
top_features_df = select_top_features(ranking_df, top_n)

output_dir.mkdir(parents=True, exist_ok=True)
summary_path = output_dir / "dataset_summary.json"
ranking_path = output_dir / "feature_ranking.csv"
top_features_path = output_dir / f"top_{top_n}_features.csv"
excluded_columns_path = output_dir / "excluded_tp53_related_features.txt"

summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
ranking_df.to_csv(ranking_path, index=False)
top_features_df.to_csv(top_features_path, index=False)
excluded_columns_path.write_text("\n".join(tp53_related_columns) + "\n", encoding="utf-8")

ranking_df.head(20)

## 7. First Model: Stratified Baseline vs Logistic Regression

Now we train on the saved top-500 feature set. The stratified dummy classifier is the sanity check: it predicts according to the training class distribution. Logistic regression is the first real model, using standardized expression values and class balancing.

Important caveat: the top-500 list was selected using the full dataset before this train/test split. These metrics are useful for checking whether there is signal, but they are not yet a final unbiased estimate. Later, feature selection should happen inside each training fold.

In [ ]:
training_results = train_top_features_model(
    expression_csv=expression_path,
    mutation_csv=mutation_path,
    top_features_csv=top_features_path,
    output_dir=model_output_dir,
    test_size=0.25,
    random_state=42,
)

training_results["metrics"]

In [ ]:
metrics = training_results["metrics"]
comparison_df = pd.DataFrame(
    [
        {"model": "dummy_stratified", **metrics["dummy_stratified"]},
        {"model": "logistic_regression_top500", **metrics["logistic_regression_top500"]},
    ]
).drop(columns=["confusion_matrix"])

comparison_df

## 8. Inspect Model Coefficients

These coefficients are not a biological conclusion by themselves, but they give us a first look at which selected genes are most influential in the linear classifier.

In [ ]:
coefficients_df = pd.read_csv(training_results["coefficients_path"])
coefficients_df.head(20)